# EJERCICIO 1: MapReduce #
Jose Manuel Ramos Chica

Esta practica esta compuesta de dos ejercicios. El primero consiste en realizar operaciones de MapReduce con Python y el segundo en usar la herramiento Hive para tratar grandes cantidades de datos.
Al final del  notebook "Notebook_ej2" hablaremos de las conclusiones de este ejercicio y como se ha usado la IA.

---

# EJERCICIO 1: MapReduce #
Primero cargaremos los datos de los ficheros usuarios.csv y eventos.csv en pandas para poder explorarlos:

In [1]:
import pandas as pd

columnas_usuarios = ['usuario_id', 'edad', 'sexo', 'pais', 'titulación', 'fecha_alta']
columnas_eventos = ['evento_id', 'usuario_id', 'fecha_hora', 'curso', 'tipo_evento', 'recurso', 'duracion_segundos', 'resultado']

df_usuarios = pd.read_csv("usuarios.csv", sep=';')
df_eventos = pd.read_csv("eventos.csv", sep=';')

In [2]:
df_usuarios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   usuario_id  15 non-null     object
 1   edad        15 non-null     int64 
 2   sexo        15 non-null     object
 3   pais        15 non-null     object
 4   titulacion  15 non-null     object
 5   fecha_alta  15 non-null     object
dtypes: int64(1), object(5)
memory usage: 848.0+ bytes


In [3]:
df_eventos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   evento_id          67 non-null     object 
 1   usuario_id         67 non-null     object 
 2   fecha_hora         67 non-null     object 
 3   curso              67 non-null     object 
 4   tipo_evento        67 non-null     object 
 5   recurso            67 non-null     object 
 6   duracion_segundos  67 non-null     int64  
 7   resultado          21 non-null     float64
dtypes: float64(1), int64(1), object(6)
memory usage: 4.3+ KB


In [4]:
df_eventos.head(3)

,evento_id,usuario_id,fecha_hora,curso,tipo_evento,recurso,duracion_segundos,resultado
0,E001,U001,2025-03-01T08:15:00,BigData,login,plataforma,120,NaN
1,E002,U001,2025-03-01T08:18:00,BigData,video,introduccion,840,NaN
2,E003,U002,2025-03-01T09:05:00,BigData,login,plataforma,95,NaN


In [5]:
df_usuarios.head(3)

,usuario_id,edad,sexo,pais,titulacion,fecha_alta
0,U001,24,F,España,Informática,2026-01-10
1,U002,31,M,España,Telecomunicación,2026-01-12
2,U003,27,F,Portugal,Matemáticas,2026-01-15


In [6]:
df_eventos['tipo_evento'].value_counts()

tipo_evento
evaluacion    21
video         16
login         15
documento      8
foro           7
Name: count, dtype: int64

Ahora, subieremos los ficheros csv a hdfs y luego procederemos a realizar el código del reduce:

In [7]:
! hdfs dfs -mkdir /ficheros_ejercicio1
! hdfs dfs -put usuarios.csv /ficheros_ejercicio1
! hdfs dfs -put eventos.csv /ficheros_ejercicio1
! hdfs dfs -ls /ficheros_ejercicio1

mkdir: `/ficheros_ejercicio1': File exists
put: `/ficheros_ejercicio1/usuarios.csv': File exists
put: `/ficheros_ejercicio1/eventos.csv': File exists
Found 7 items
-rw-r--r--   3 root supergroup       4373 2026-11-11 10:39 /ficheros_ejercicio1/eventos.csv
drwxr-xr-x   - root supergroup          0 2026-11-13 13:00 /ficheros_ejercicio1/salida
drwxr-xr-x   - root supergroup          0 2026-11-13 13:01 /ficheros_ejercicio1/salida_ej2
drwxr-xr-x   - root supergroup          0 2026-11-13 13:01 /ficheros_ejercicio1/salida_ej2_join
drwxr-xr-x   - root supergroup          0 2026-11-13 13:01 /ficheros_ejercicio1/salida_ej3
drwxr-xr-x   - root supergroup          0 2026-11-13 13:01 /ficheros_ejercicio1/salida_ej3_join
-rw-r--r--   3 root supergroup        719 2026-11-11 10:39 /ficheros_ejercicio1/usuarios.csv


In [8]:
! mkdir ejercicio1

mkdir: cannot create directory ‘ejercicio1’: File exists


---
## EJERCICIO 1.1: Contador de eventos ##
Ahora, procederemos a realizar los ficheros python que se encargaran de realizar las funciones de mapreduce

Primero realizaremos el mapper, el cual se encargara de extraer el valor que necesitamos de la entrada (en este caso, la variable `tipo_evento`).
Antes de eso, comprobamos que el valor de la variable no sea el propio nombre de la columna. Esto es debido a que la primera fila contiene los nombres de cada columna.

In [9]:
%%writefile ejercicio1/mapper_ej1.py
#!/usr/bin/env python3
import sys

for linea in sys.stdin:
    splitted = linea.strip().split(';')
    if(splitted[0]=="evento_id"):
        continue
    tipo = splitted[4]
    print(f"{tipo}")

Overwriting ejercicio1/mapper_ej1.py


En el reducer simplemente servira para contar cuantas veces aparece cada valor de `tipo_evento`. Para ello, crearemos un diccionario en el cual la *key* sera el valor de la variable y el *value* el número de veces que aparece.

In [10]:
%%writefile ejercicio1/reducer_ej1.py
#!/usr/bin/env python3
import sys

result = {}

for linea in sys.stdin:
    tipo = linea.strip()
    if not tipo:
        continue
    result[tipo] = result.get(tipo,0) + 1 

for key,value in result.items():
    print(f"{key}\t{value}")
    

Overwriting ejercicio1/reducer_ej1.py


In [11]:
! hdfs dfs -rm -r -f /ficheros_ejercicio1/salida
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files ejercicio1/mapper_ej1.py,ejercicio1/reducer_ej1.py \
-mapper "python3 mapper_ej1.py" \
-reducer "python3 reducer_ej1.py" \
-input /ficheros_ejercicio1/eventos.csv \
-output /ficheros_ejercicio1/salida/eventos_tipo.csv

Deleted /ficheros_ejercicio1/salida
packageJobJar: [/tmp/hadoop-unjar4013102029089021048/] [] /tmp/streamjob8699044347370302333.jar tmpDir=null
2026-11-13 13:18:49,144 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-13 13:18:49,243 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-13 13:18:49,390 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794560170642_0041
2026-11-13 13:18:49,753 INFO mapred.FileInputFormat: Total input files to process : 1
2026-11-13 13:18:49,810 INFO mapreduce.JobSubmitter: number of splits:2
2026-11-13 13:18:49,896 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794560170642_0041
2026-11-13 13:18:49,896 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-13 13:18:50,018 INFO conf.Configuration: resource-types.xml not found
2026-11-13 13:1

In [12]:
! hdfs dfs -cat /ficheros_ejercicio1/salida/eventos_tipo.csv/part-*

documento	8
evaluacion	21
foro	7
login	15
video	16


In [13]:
df_eventos.groupby('tipo_evento').size()

tipo_evento
documento      8
evaluacion    21
foro           7
login         15
video         16
dtype: int64

El MapReduce funciona correctamente ya que devuelve el mismo resultado que si la operación se hubiera realizado usando pandas.

Visto esto, el número de MapReduce necesarios para esta tarea es solo 1: un mapper extraer la información y un reducer para agrupar y contar.

---
## EJERCICIO 1.2: Contador de evento por pais ##
La tarea ahora es mejorar el *MapReduce* anterior para que devuelva el número de eventos realizado por usuarios de cada país. Para ello es necesario reunir ambos ficheros, ya que el fichero de eventos almacena el identificador de usuario (usuario_id), y el país se almacena solo en el fichero de usuarios.


Lo primero que crearemos sera el mapper. Este se encargara de distinguir de que tabla es el entrada que esta tratando (lo hara comprobando la longitud de la lista de variables que crearemos haciendo una opración de *split*). Una vez hecho esto, extraeremos el id del usuario (el cual, es el identificador común en ambas tablas) y lo mandaremos como salida junto a una etiqueta (U o E) la cual se encargara de identificar a que tipo de entrada pertenece dicha salida. En el caso de que la entrada sea de la tabla usuarios, también mandara el valor de la variable *país*.

In [14]:
%%writefile ejercicio1/mapper_ej2_join.py
#!/usr/bin/env python3
import sys

for linea in sys.stdin:
    campos = linea.strip().split(';')

    if campos[0] in ("usuario_id", "evento_id"):
        continue

    if len(campos) == 6:          # usuarios.csv
        usuario_id = campos[0]
        pais = campos[3].strip()
        print(f"{usuario_id}\tU\t{pais}")

    elif len(campos) == 8:        # eventos.csv
        usuario_id = campos[1]
        print(f"{usuario_id}\tE")

Overwriting ejercicio1/mapper_ej2_join.py


Tras la fase de *mapper*, el mecanismo *shuffle & sort* ordena las salidas en función del ID de usuario (ya que es la variable en la primera posición de la cadena). De esta manera, la entrada de eventos y la entrada que usaremos para saber el país del usuario apareceran de forma contigua y facilitaran el trabajo al siguiente mapper, el cual, se encargara de hacer el join. 

El *reducer* simplemente se encargara de hacer el recuento de eventos en los que ha participado el usuario (con las entradas que tengan la etiqueta *E*) y asignar el país con las entradas que tengan la etiqueta *U*. Una vez haya recopilado toda la información relacionada con este usuario y el siguiente ID de usuario introducido pertenezca a otro usuario, enviara como salida el país del usuario y el número de eventos en los que participó dicho usuario. 

In [15]:
%%writefile ejercicio1/reducer_ej2_join.py
#!/usr/bin/env python3
import sys

usuario_actual = None
pais = None
n_eventos = 0

def emitir(pais, n_eventos):
    if n_eventos > 0:
        print(f"{pais if pais else 'desconocido'}\t{n_eventos}")

for linea in sys.stdin:
    campos = linea.strip().split('\t')
    if len(campos) < 2:
        continue
    usuario_id, etiqueta = campos[0], campos[1]

    # Cambio de usuario: emitir el anterior y reiniciar
    if usuario_id != usuario_actual:
        if usuario_actual is not None:
            emitir(pais, n_eventos)
        usuario_actual = usuario_id
        pais = None
        n_eventos = 0

    if etiqueta == "U":
        pais = campos[2]
    elif etiqueta == "E":
        n_eventos += 1

# último usuario
if usuario_actual is not None:
    emitir(pais, n_eventos)

Overwriting ejercicio1/reducer_ej2_join.py


El *reducer* anterior emitira pais y número de eventos por cada uno de los usuarios. El siguiente se encargara de sumar el total de eventos de todos los usuarios en función de su pais.

In [16]:
%%writefile ejercicio1/reducer_ej2_suma.py
#!/usr/bin/env python3
import sys

result = {}

for linea in sys.stdin:
    campos = linea.strip().split('\t')
    if len(campos) != 2:
        continue
    pais, n = campos
    result[pais] = result.get(pais, 0) + int(n)

for pais, total in result.items():
    print(f"{pais}\t{total}")

Overwriting ejercicio1/reducer_ej2_suma.py


In [17]:
! hdfs dfs -rm -r -f /ficheros_ejercicio1/salida_ej2_join
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -files ejercicio1/mapper_ej2_join.py,ejercicio1/reducer_ej2_join.py \
    -mapper "python3 mapper_ej2_join.py" \
    -reducer "python3 reducer_ej2_join.py" \
    -input /ficheros_ejercicio1/usuarios.csv \
    -input /ficheros_ejercicio1/eventos.csv \
    -output /ficheros_ejercicio1/salida_ej2_join

Deleted /ficheros_ejercicio1/salida_ej2_join
packageJobJar: [/tmp/hadoop-unjar5967784702637347824/] [] /tmp/streamjob5182188283858590567.jar tmpDir=null
2026-11-13 13:19:24,330 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-13 13:19:24,428 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-13 13:19:24,572 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794560170642_0043
2026-11-13 13:19:24,810 INFO mapred.FileInputFormat: Total input files to process : 2
2026-11-13 13:19:24,851 INFO mapreduce.JobSubmitter: number of splits:3
2026-11-13 13:19:24,922 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794560170642_0043
2026-11-13 13:19:24,922 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-13 13:19:25,038 INFO conf.Configuration: resource-types.xml not found
2026-1

In [18]:
! hdfs dfs -rm -r -f /ficheros_ejercicio1/salida_ej2
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -files ejercicio1/reducer_ej2_suma.py \
    -mapper cat \
    -reducer "python3 reducer_ej2_suma.py" \
    -input /ficheros_ejercicio1/salida_ej2_join \
    -output /ficheros_ejercicio1/salida_ej2

Deleted /ficheros_ejercicio1/salida_ej2
packageJobJar: [/tmp/hadoop-unjar7448969162204667243/] [] /tmp/streamjob1079654721235923844.jar tmpDir=null
2026-11-13 13:20:02,909 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-13 13:20:03,005 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-13 13:20:03,139 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794560170642_0045
2026-11-13 13:20:03,363 INFO mapred.FileInputFormat: Total input files to process : 1
2026-11-13 13:20:03,398 INFO mapreduce.JobSubmitter: number of splits:2
2026-11-13 13:20:03,467 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794560170642_0045
2026-11-13 13:20:03,467 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-13 13:20:03,576 INFO conf.Configuration: resource-types.xml not found
2026-11-13 

In [19]:
! hdfs dfs -cat /ficheros_ejercicio1/salida_ej2/part-*

Argentina	4
Chile	4
Colombia	5
España	36
México	5
Perú	4
Portugal	9


In [20]:
resultado = (
    df_eventos
    .merge(df_usuarios[['usuario_id', 'pais']], on='usuario_id', how='left')
    .groupby('pais', dropna=False)
    .size()
    .reset_index(name='numero_eventos')
)
resultado

,pais,numero_eventos
0,Argentina,4
1,Chile,4
2,Colombia,5
3,España,36
4,México,5
5,Perú,4
6,Portugal,9


Como podemos comprobar, funciona perfectamente.

Para esta tarea han sido necesarias dos MapReduce:
1. Uno que una cada evento con el país de su usuario. La clave común a los dos ficheros es `usuario_id`, así que este job agrupa por usuario.
2. Otro que sume los eventos por país.


---

## EJERCICIO 1.3: Mejorando el contador de eventos por país ##
Ahora, partiendo de los ejercicios anteriores, hay que hacer para que cada país y tipo de evento calcule la duración media de los eventos expresada en segundos.

El mapper prácticamente igual. Solo cambiará que, cuando la entrada sea de la tabla eventos, también añadiremos la variable *duracion* a la salida.

In [21]:
%%writefile ejercicio1/mapper_ej3_join.py
#!/usr/bin/env python3
import sys

for linea in sys.stdin:
    campos = linea.strip().split(';')

    if campos[0] in ("usuario_id", "evento_id"):
        continue

    if len(campos) == 6:          # usuarios.csv
        usuario_id = campos[0]
        pais = campos[3].strip()
        print(f"{usuario_id}\tU\t{pais}")

    elif len(campos) == 8:        # eventos.csv
        usuario_id = campos[1]
        tipo = campos[4].strip()
        duracion = campos[6].strip()
        if duracion.isdigit():
            print(f"{usuario_id}\tE\t{tipo}\t{duracion}")

Overwriting ejercicio1/mapper_ej3_join.py


Al primer *reducer* le añadiremos que no solo debe de guardar el número de eventos, sino que también debe de realizar un sumatorio de su duración y mandarlo en la salida.

In [22]:
%%writefile ejercicio1/reducer_ej3_join.py
#!/usr/bin/env python3
import sys

usuario_actual = None
pais = None
por_tipo = {}          # {tipo: [suma_duracion, n_eventos]}

def emitir(pais, por_tipo):
    for tipo, (suma, n) in por_tipo.items():
        print(f"{pais if pais else 'desconocido'}\t{tipo}\t{suma}\t{n}")

for linea in sys.stdin:
    campos = linea.strip().split('\t')
    if len(campos) < 2:
        continue
    usuario_id, etiqueta = campos[0], campos[1]

    if usuario_id != usuario_actual:
        if usuario_actual is not None:
            emitir(pais, por_tipo)
        usuario_actual = usuario_id
        pais = None
        por_tipo = {}

    if etiqueta == "U":
        pais = campos[2]
    elif etiqueta == "E" and len(campos) == 4:
        tipo, duracion = campos[2], int(campos[3])
        acum = por_tipo.setdefault(tipo, [0, 0])
        acum[0] += duracion
        acum[1] += 1

if usuario_actual is not None:
    emitir(pais, por_tipo)

Overwriting ejercicio1/reducer_ej3_join.py


En el último reducer haremos un recopilatorio de la duración media de cada país y tipo de evento y mandaremos el resultado por la salida.
Para esto, crearemos un diccionario:
- La *clave* sera una tupla país-tipo
- El *valor* sera una lista donde el primer valor sera la duración y el segundo el número de eventos

In [23]:
%%writefile ejercicio1/reducer_ej3_media.py
#!/usr/bin/env python3
import sys

result = {}            # {(pais, tipo): [suma_duracion, n_eventos]}

for linea in sys.stdin:
    campos = linea.strip().split('\t')
    if len(campos) != 4:
        continue
    pais, tipo, suma, n = campos
    acum = result.setdefault((pais, tipo), [0, 0])
    acum[0] += int(suma)
    acum[1] += int(n)

for (pais, tipo), (suma, n) in sorted(result.items()):
    print(f"{pais}\t{tipo}\t{suma / n:.2f}")

Overwriting ejercicio1/reducer_ej3_media.py


In [24]:
! hdfs dfs -rm -r -f /ficheros_ejercicio1/salida_ej3_join
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -files ejercicio1/mapper_ej3_join.py,ejercicio1/reducer_ej3_join.py \
    -mapper "python3 mapper_ej3_join.py" \
    -reducer "python3 reducer_ej3_join.py" \
    -input /ficheros_ejercicio1/usuarios.csv \
    -input /ficheros_ejercicio1/eventos.csv \
    -output /ficheros_ejercicio1/salida_ej3_join

Deleted /ficheros_ejercicio1/salida_ej3_join
packageJobJar: [/tmp/hadoop-unjar5412150924653271466/] [] /tmp/streamjob4453236866527917264.jar tmpDir=null
2026-11-13 13:20:44,013 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-13 13:20:44,121 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-13 13:20:44,282 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794560170642_0046
2026-11-13 13:20:44,575 INFO mapred.FileInputFormat: Total input files to process : 2
2026-11-13 13:20:44,621 INFO mapreduce.JobSubmitter: number of splits:3
2026-11-13 13:20:44,696 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794560170642_0046
2026-11-13 13:20:44,696 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-13 13:20:44,830 INFO conf.Configuration: resource-types.xml not found
2026-1

In [25]:
! hdfs dfs -rm -r -f /ficheros_ejercicio1/salida_ej3
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -files ejercicio1/reducer_ej3_media.py \
    -mapper cat \
    -reducer "python3 reducer_ej3_media.py" \
    -input /ficheros_ejercicio1/salida_ej3_join \
    -output /ficheros_ejercicio1/salida_ej3

Deleted /ficheros_ejercicio1/salida_ej3
packageJobJar: [/tmp/hadoop-unjar6207678567900251909/] [] /tmp/streamjob4762854218395260159.jar tmpDir=null
2026-11-13 13:21:00,846 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-13 13:21:00,960 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-13 13:21:01,121 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794560170642_0048
2026-11-13 13:21:01,371 INFO mapred.FileInputFormat: Total input files to process : 1
2026-11-13 13:21:01,407 INFO mapreduce.JobSubmitter: number of splits:2
2026-11-13 13:21:01,485 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794560170642_0048
2026-11-13 13:21:01,485 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-13 13:21:01,605 INFO conf.Configuration: resource-types.xml not found
2026-11-13 

In [26]:
! hdfs dfs -cat /ficheros_ejercicio1/salida_ej3/part-*

Argentina	documento	540.00
Argentina	evaluacion	910.00
Argentina	login	100.00
Argentina	video	900.00
Chile	documento	510.00
Chile	evaluacion	920.00
Chile	login	130.00
Chile	video	960.00
Colombia	evaluacion	895.00
Colombia	foro	720.00
Colombia	login	115.00
Colombia	video	840.00
España	documento	540.00
España	evaluacion	895.45
España	foro	532.50
España	login	93.75
España	video	903.33
México	evaluacion	920.00
México	foro	540.00
México	login	105.00
México	video	1200.00
Perú	evaluacion	930.00
Perú	foro	660.00
Perú	login	90.00
Perú	video	870.00
Portugal	documento	525.00
Portugal	evaluacion	913.33
Portugal	login	117.50
Portugal	video	975.00


In [27]:
df_join = df_eventos.merge(df_usuarios[['usuario_id', 'pais']], on='usuario_id', how='left')

df_join['duracion_segundos'] = pd.to_numeric(df_join['duracion_segundos'], errors='coerce')

resultado = (
    df_join
    .groupby(['pais', 'tipo_evento'], dropna=False)['duracion_segundos']
    .mean()
    .round(2)
    .reset_index(name='duracion_media')
)
resultado

,pais,tipo_evento,duracion_media
0,Argentina,documento,540.00
1,Argentina,evaluacion,910.00
2,Argentina,login,100.00
3,Argentina,video,900.00
4,Chile,documento,510.00
5,Chile,evaluacion,920.00
6,Chile,login,130.00
7,Chile,video,960.00
8,Colombia,evaluacion,895.00
9,Colombia,foro,720.00


**¿Cuántos pasos MapReduce son necesarios?** Dos, con la misma estructura que en el 1.2:

1. **Job 1 (join):** agrupa por `usuario_id` para asignar a cada evento el país de su usuario, y acumula por tipo de evento la suma de duraciones y el número de eventos.
2. **Job 2 (agregación):** junta los resultados parciales de todos los usuarios y calcula la duración media para cada par (país, tipo de evento).